In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- attr_getter_unseen_frame ---
FIX_ATTR_GETTER_UNSEEN_FRAME_COMMON_CITATIONS_SCORES = [(101, 0.9), (102, 0.7), (103, 0.5)]
FIX_ATTR_GETTER_UNSEEN_FRAME_SORT_DOCUMENT_SCORES = lambda scores: sorted(scores, key=lambda x: x[1], reverse=True)

# --- attr_getter_unseen_iter ---
FIX_ATTR_GETTER_UNSEEN_ITER_CANDIDATE_CITATION_URLS = {0: "https://example.com/1", 1: "https://example.com/2"}
FIX_ATTR_GETTER_UNSEEN_ITER_D3_DOCUMENT_ID = 0

print("✅ Fixtures loaded")
self = SimpleNamespace(
    documents_data=pd.DataFrame({
        "document_id":[101,102,103],
        "title":["A","B","C"],
        "abstract":["X","Y","Z"],
        "publication_date":pd.to_datetime(["2021-01-01","2022-01-01","2023-01-01"]),
        "citations":[{101: "https://a.com"}, {102: "https://b.com"}, {}],
    })
)


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_attr_getter_unseen_frame(common_citations_scores, sort_document_scores):
    return pd.DataFrame(
        {"scores": [sort_document_scores(common_citations_scores)]}, index=[-1]
    ).rename_axis("document_id", axis="index")
    return None

def before_attr_getter_unseen_iter(candidate_citation_urls, d3_document_id):
    for d3_document_id, candidate_citation_urls in self.documents_data["citations"].items():
        pass
    return None

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_attr_getter_unseen_frame(common_citations_scores, sort_document_scores):

    return pl.DataFrame(
        {"scores": [sort_document_scores(common_citations_scores)]}
    )

def gen_attr_getter_unseen_iter(candidate_citation_urls, d3_document_id):
    for d3_document_id, candidate_citation_urls in self.documents_data["citations"].items():
        pass
    return None

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: attr_getter_unseen_frame ===

# L1 smoke – generated
try:
    _r = gen_attr_getter_unseen_frame(FIX_ATTR_GETTER_UNSEEN_FRAME_COMMON_CITATIONS_SCORES, FIX_ATTR_GETTER_UNSEEN_FRAME_SORT_DOCUMENT_SCORES)
    print("✅ L1 smoke gen_attr_getter_unseen_frame: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_attr_getter_unseen_frame: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_attr_getter_unseen_frame(FIX_ATTR_GETTER_UNSEEN_FRAME_COMMON_CITATIONS_SCORES, FIX_ATTR_GETTER_UNSEEN_FRAME_SORT_DOCUMENT_SCORES)
    print("✅ L1 smoke before_attr_getter_unseen_frame: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_attr_getter_unseen_frame: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_attr_getter_unseen_frame(FIX_ATTR_GETTER_UNSEEN_FRAME_COMMON_CITATIONS_SCORES, FIX_ATTR_GETTER_UNSEEN_FRAME_SORT_DOCUMENT_SCORES)
    _rg = gen_attr_getter_unseen_frame(FIX_ATTR_GETTER_UNSEEN_FRAME_COMMON_CITATIONS_SCORES, FIX_ATTR_GETTER_UNSEEN_FRAME_SORT_DOCUMENT_SCORES)
    compare(_rb, _rg, "attr_getter_unseen_frame")
except Exception as _e:
    print(f"❌ L2 equivalence attr_getter_unseen_frame: setup error — {type(_e).__name__}: {_e}")

# L3 edge - no citation-score rows still returns the sentinel document row.
try:
    _empty_scores = []
    _rb = before_attr_getter_unseen_frame(
        _empty_scores, FIX_ATTR_GETTER_UNSEEN_FRAME_SORT_DOCUMENT_SCORES
    )
    _rg = gen_attr_getter_unseen_frame(
        _empty_scores, FIX_ATTR_GETTER_UNSEEN_FRAME_SORT_DOCUMENT_SCORES
    )
    compare(_rb, _rg, "L3 edge attr_getter_unseen_frame empty scores", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge attr_getter_unseen_frame empty scores: {type(_e).__name__}: {_e}")
